## DATA PREPROCESSING - ENTRY LEVEL / INTERNSHIP STYLE

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

In [2]:
# STEP 0: LOAD DATA

df = pd.read_csv('real_company_raw_data.csv')
print(f"Original shape: {df.shape}")

Original shape: (2070, 14)


In [3]:
# STEP 1: REMOVE DUPLICATES
print("STEP 1: REMOVE DUPLICATES\n")
print(f"Duplicate rows before: {df.duplicated().sum()}")
print(f"Duplicate Customer IDs before: {df['Customer ID'].duplicated().sum()}")

df = df.drop_duplicates(keep='first')
df = df.drop_duplicates(subset=['Customer ID'], keep='first')
df = df.copy()

print(f"Shape after removing duplicates: {df.shape}")

STEP 1: REMOVE DUPLICATES

Duplicate rows before: 45
Duplicate Customer IDs before: 45
Shape after removing duplicates: (2025, 14)


<h3>Notes:</h3>

<b style="color:blue;">df = df.drop_duplicates(keep='first')</b><br>
Removes rows that are exactly the same and keeps the first one.<br>

<b>Example:</b> If the same row appears twice in the dataset, the second copy is removed.<br>

<b style="color:blue;">df = df.drop_duplicates(subset=['Customer ID'], keep='first')</b><br>
Checks for duplicate <b>Customer ID</b> values and keeps only the first record for each customer.<br>

<b>Example:</b> If <b>Customer ID C001</b> appears three times, only the first record is kept and the other two are removed.<br>

<b>Important:</b> Use this only when each customer should have one record in the dataset. If a customer can have multiple orders or transactions, do not use this method because those records may be valid.

In [4]:
# STEP 2: CLEAN COLUMN NAMES
print("STEP 2: CLEAN COLUMN NAMES\n")
print(f"Old columns: {list(df.columns)}")

df.columns = (
    df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('(', '')
        .str.replace(')', '')
)
print(f"New columns: {list(df.columns)}")


STEP 2: CLEAN COLUMN NAMES

Old columns: ['Customer ID', 'Full Name', 'Email Address', 'Phone Number', 'Registration Date', 'Purchase Amount', 'Product Category', 'Satisfaction Score (1-5)', 'Country', 'Customer Age', 'Gender', 'Order Status', 'Discount Applied', 'Shipping Cost']
New columns: ['customer_id', 'full_name', 'email_address', 'phone_number', 'registration_date', 'purchase_amount', 'product_category', 'satisfaction_score_1-5', 'country', 'customer_age', 'gender', 'order_status', 'discount_applied', 'shipping_cost']


In [5]:
# STEP 3: CLEAN FULL NAME
print("STEP 3: CLEAN FULL NAME\n")
before = df['full_name'].isna().sum()

df['full_name'] = df['full_name'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
bad_names = df['full_name'].str.contains(r'[@0-9]', regex=True, na=False)
df.loc[bad_names, 'full_name'] = np.nan

print(f"Names with digits or @ symbols found: {bad_names.sum()}")
print(f"Names set to NaN: {df['full_name'].isna().sum() - before}")
print(f"Sample valid names: {df['full_name'].dropna().head(3).tolist()}")

STEP 3: CLEAN FULL NAME

Names with digits or @ symbols found: 20
Names set to NaN: 20
Sample valid names: ['Donald Brown', 'David Garcia', 'Robert Wilson']


In [6]:
# STEP 4: CLEAN EMAIL
print("STEP 4: CLEAN EMAIL\n")
before = df['email_address'].isna().sum()

df['email_address'] = df['email_address'].astype(str).str.strip().str.replace(r'\s+', '', regex=True).str.lower()
df.loc[df['email_address'] == 'nan', 'email_address'] = np.nan

email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
valid_email = df['email_address'].str.match(email_pattern, na=False)
invalid = (~valid_email) & df['email_address'].notna()
df.loc[~valid_email, 'email_address'] = np.nan

placeholder = df['email_address'].str.contains(r'john\.smith@|user\d+@', na=False)
df.loc[placeholder, 'email_address'] = np.nan

print(f"Invalid emails found: {invalid.sum()}")
print(f"Placeholder emails found: {placeholder.sum()}")
print(f"Total emails set to NaN: {df['email_address'].isna().sum() - before}")
print(f"Valid emails remaining: {df['email_address'].notna().sum()}")

STEP 4: CLEAN EMAIL

Invalid emails found: 419
Placeholder emails found: 152
Total emails set to NaN: 571
Valid emails remaining: 1251


In [7]:
# STEP 5: CLEAN PHONE NUMBER
print("STEP 5: CLEAN PHONE NUMBER\n")
before = df['phone_number'].isna().sum()

df['phone_number'] = df['phone_number'].astype(str).str.strip()
df.loc[df['phone_number'] == 'nan', 'phone_number'] = np.nan
df['phone_number'] = df['phone_number'].str.replace(r'\D', '', regex=True)

too_short = (df['phone_number'].str.len() < 10) & df['phone_number'].notna()
too_long = (df['phone_number'].str.len() > 15) & df['phone_number'].notna()
df.loc[too_short | too_long, 'phone_number'] = np.nan

print(f"Phone numbers too short (<10 digits): {too_short.sum()}")
print(f"Phone numbers too long (>15 digits): {too_long.sum()}")
print(f"Total phones set to NaN: {df['phone_number'].isna().sum() - before}")
print(f"Valid phones remaining: {df['phone_number'].notna().sum()}")


STEP 5: CLEAN PHONE NUMBER

Phone numbers too short (<10 digits): 73
Phone numbers too long (>15 digits): 0
Total phones set to NaN: 73
Valid phones remaining: 1796


In [8]:
# STEP 6: CLEAN REGISTRATION DATE
print("STEP 6: CLEAN REGISTRATION DATE\n")
before = df['registration_date'].isna().sum()

df['registration_date'] = pd.to_datetime(df['registration_date'], errors='coerce')
if df['registration_date'].dtype == object:
    df['registration_date'] = pd.to_datetime(df['registration_date'], errors='coerce')

current_date = pd.Timestamp('2026-08-20')
future = df['registration_date'] > current_date
too_old = df['registration_date'] < pd.Timestamp('2000-01-01')
df.loc[future | too_old, 'registration_date'] = np.nan

print(f"Future dates (> 2026-08-20): {future.sum()}")
print(f"Dates too old (< 2000-01-01): {too_old.sum()}")
print(f"Total dates set to NaN: {df['registration_date'].isna().sum() - before}")
if df['registration_date'].notna().any():
    print(f"Date range: {df['registration_date'].min()} to {df['registration_date'].max()}")


STEP 6: CLEAN REGISTRATION DATE

Future dates (> 2026-08-20): 0
Dates too old (< 2000-01-01): 0
Total dates set to NaN: 227
Date range: 2018-01-01 00:00:00 to 2024-11-05 00:00:00


In [9]:
# STEP 7: CLEAN PURCHASE AMOUNT
print("STEP 7: CLEAN PURCHASE AMOUNT\n")
before = df['purchase_amount'].isna().sum()

df['purchase_amount'] = df['purchase_amount'].astype(str).str.strip()
free_mask = df['purchase_amount'].str.lower() == 'free'
df.loc[free_mask, 'purchase_amount'] = '0'
print(f"'Free' values converted to 0: {free_mask.sum()}")

df['purchase_amount'] = df['purchase_amount'].str.replace(r'[\$\s,]|USD|EUR', '', regex=True)
df['purchase_amount'] = pd.to_numeric(df['purchase_amount'], errors='coerce')

neg = df['purchase_amount'] < 0
df.loc[neg, 'purchase_amount'] = np.nan
print(f"Negative values set to NaN: {neg.sum()}")

Q1, Q3 = df['purchase_amount'].quantile([0.25, 0.75])
IQR = Q3 - Q1
if pd.notna(IQR) and IQR > 0:
    upper = Q3 + 1.5 * IQR
    out = df['purchase_amount'] > upper
    df.loc[out, 'purchase_amount'] = np.nan
    print(f"Outliers (>{upper:.2f}) set to NaN: {out.sum()}")

print(f"Total purchase amounts set to NaN: {df['purchase_amount'].isna().sum() - before}")
print(f"Min: {df['purchase_amount'].min()}, Max: {df['purchase_amount'].max()}")


STEP 7: CLEAN PURCHASE AMOUNT

'Free' values converted to 0: 28
Negative values set to NaN: 0
Outliers (>1250.82) set to NaN: 66
Total purchase amounts set to NaN: 87
Min: 0.0, Max: 1197.0


In [10]:
# STEP 8: CLEAN SHIPPING COST
print("STEP 8: CLEAN SHIPPING COST\n")
before = df['shipping_cost'].isna().sum()

df['shipping_cost'] = df['shipping_cost'].astype(str).str.strip().str.replace(r'[\$\s,]|USD|EUR', '', regex=True)
df['shipping_cost'] = pd.to_numeric(df['shipping_cost'], errors='coerce')

neg = df['shipping_cost'] < 0
df.loc[neg, 'shipping_cost'] = np.nan
print(f"Negative values set to NaN: {neg.sum()}")

Q1s, Q3s = df['shipping_cost'].quantile([0.25, 0.75])
if pd.notna(Q1s) and pd.notna(Q3s) and (Q3s - Q1s) > 0:
    upper = Q3s + 1.5 * (Q3s - Q1s)
    out = df['shipping_cost'] > upper
    df.loc[out, 'shipping_cost'] = np.nan
    print(f"Outliers (>{upper:.2f}) set to NaN: {out.sum()}")

print(f"Total shipping costs set to NaN: {df['shipping_cost'].isna().sum() - before}")
print(f"Min: {df['shipping_cost'].min()}, Max: {df['shipping_cost'].max()}")

STEP 8: CLEAN SHIPPING COST

Negative values set to NaN: 0
Outliers (>71.67) set to NaN: 0
Total shipping costs set to NaN: 0
Min: 5.02, Max: 49.97


In [11]:
# STEP 9: CLEAN CUSTOMER AGE
print("STEP 9: CLEAN CUSTOMER AGE\n")
before = df['customer_age'].isna().sum()

df['customer_age'] = df['customer_age'].astype(str).str.strip()
bad_ages = ['n/a', 'twenty-five', 'twenty five', 'thirty', 'years', '30 years']
text_fixed = 0
for bad in bad_ages:
    mask = df['customer_age'].str.lower() == bad
    if mask.sum() > 0:
        print(f"  '{bad}' found and cleaned: {mask.sum()}")
        text_fixed += mask.sum()
    df.loc[mask, 'customer_age'] = np.nan

df['customer_age'] = pd.to_numeric(df['customer_age'], errors='coerce')
invalid_age = (df['customer_age'] < 18) | (df['customer_age'] > 120)
df.loc[invalid_age, 'customer_age'] = np.nan

print(f"Ages < 18 or > 120 set to NaN: {invalid_age.sum()}")
print(f"Text typos cleaned: {text_fixed}")
print(f"Total ages set to NaN: {df['customer_age'].isna().sum() - before}")
print(f"Age range: {df['customer_age'].min()} to {df['customer_age'].max()}")

STEP 9: CLEAN CUSTOMER AGE

  'twenty-five' found and cleaned: 17
  '30 years' found and cleaned: 16
Ages < 18 or > 120 set to NaN: 36
Text typos cleaned: 33
Total ages set to NaN: 69
Age range: 18.0 to 120.0


In [12]:
# STEP 10: CLEAN SATISFACTION SCORE
print("STEP 10: CLEAN SATISFACTION SCORE\n")
before = df['satisfaction_score_1-5'].isna().sum()

df['satisfaction_score_1-5'] = df['satisfaction_score_1-5'].astype(str).str.strip().str.lower()

score_map = {'bad': 1, 'poor': 1, 'good': 4, 'excellent': 5}
text_fixed = 0
for k, v in score_map.items():
    mask = df['satisfaction_score_1-5'] == k
    if mask.sum() > 0:
        print(f"  '{k}' mapped to {v}: {mask.sum()}")
        text_fixed += mask.sum()
    df.loc[mask, 'satisfaction_score_1-5'] = v

df['satisfaction_score_1-5'] = pd.to_numeric(df['satisfaction_score_1-5'], errors='coerce')
invalid_score = (df['satisfaction_score_1-5'] < 1) | (df['satisfaction_score_1-5'] > 5)
df.loc[invalid_score, 'satisfaction_score_1-5'] = np.nan

print(f"Scores outside 1-5 set to NaN: {invalid_score.sum()}")
print(f"Text mappings cleaned: {text_fixed}")
print(f"Total scores set to NaN: {df['satisfaction_score_1-5'].isna().sum() - before}")
print(f"Valid scores: {sorted(df['satisfaction_score_1-5'].dropna().unique())}")


STEP 10: CLEAN SATISFACTION SCORE

  'bad' mapped to 1: 21
  'good' mapped to 4: 11
  'excellent' mapped to 5: 15
Scores outside 1-5 set to NaN: 88
Text mappings cleaned: 47
Total scores set to NaN: 88
Valid scores: [np.float64(1.0), np.float64(1.2), np.float64(1.4), np.float64(1.5), np.float64(1.7), np.float64(1.8), np.float64(1.9), np.float64(2.0), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.6), np.float64(2.7), np.float64(2.9), np.float64(3.0), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(3.5), np.float64(3.6), np.float64(3.8), np.float64(4.0), np.float64(4.2), np.float64(4.4), np.float64(4.8), np.float64(5.0)]


<h3>Notes:</h3>

<b style="color:#2E7D32;">mask</b> is a True/False condition used to find the rows that match a specific value.<br>

<b>Example:</b> If the satisfaction scores are 
<b style="color:#1565C0;">['good', 'bad', 'excellent', 'good']</b>, 
then this code:<br><br>

<b style="color:#2E7D32;">mask = df['satisfaction_score_1-5'] == 'good'</b><br>

checks each row and returns:<br>

<b style="color:#1565C0;">[True, False, False, True]</b><br>

<b>True</b> means the value matches <b>'good'</b>, while <b>False</b> means it does not.<br>

Then:<br>

<b style="color:#2E7D32;">df.loc[mask, 'satisfaction_score_1-5'] = 4</b><br>

changes only the rows where <b>mask</b> is <b>True</b>.<br>

<b style="color:#C62828;">In simple words:</b> <b>mask</b> helps us find the exact rows we want to change without affecting the other rows.

In [13]:
# STEP 11: STANDARDIZE COUNTRY
print("STEP 11: STANDARDIZE COUNTRY\n")
print(f"Unique countries BEFORE: {df['country'].dropna().unique().tolist()}")

df['country'] = df['country'].astype(str).str.strip().str.lower()
country_map = {
    'usa': 'United States', 'us': 'United States', 'united states': 'United States',
    'uk': 'United Kingdom', 'united kingdom': 'United Kingdom',
    'fr': 'France', 'france': 'France', 'de': 'Germany', 'germany': 'Germany',
    'can': 'Canada', 'canada': 'Canada', 'aus': 'Australia', 'australia': 'Australia',
    'jp': 'Japan', 'japan': 'Japan', 'nan': np.nan
}
df['country'] = df['country'].map(country_map)
print(f"Unique countries AFTER: {df['country'].dropna().unique().tolist()}")

STEP 11: STANDARDIZE COUNTRY

Unique countries BEFORE: ['Australia', 'can', 'United Kingdom', 'FR', 'Germany', 'UK', 'Canada', 'USA', 'JP', 'Japan', 'France', 'CAN', 'AUS', 'DE', 'US', 'United States', 'de', 'united states', 'aus', 'japan', 'australia', 'us', 'germany', 'usa', 'fr', 'uk', 'united kingdom', 'france', 'jp', 'canada']
Unique countries AFTER: ['Australia', 'Canada', 'United Kingdom', 'France', 'Germany', 'United States', 'Japan']


In [14]:
# STEP 12: STANDARDIZE GENDER
print("STEP 12: STANDARDIZE GENDER\n")
print(f"Unique genders BEFORE: {df['gender'].dropna().unique().tolist()}")

df['gender'] = df['gender'].astype(str).str.strip().str.lower()
gender_map = {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female', 'nan': np.nan}
df['gender'] = df['gender'].map(gender_map)
print(f"Unique genders AFTER: {df['gender'].dropna().unique().tolist()}")


STEP 12: STANDARDIZE GENDER

Unique genders BEFORE: ['Female', 'Male', 'F', 'male', 'MALE', 'FEMALE', 'female', 'M']
Unique genders AFTER: ['Female', 'Male']


In [15]:
# STEP 13: STANDARDIZE ORDER STATUS
print("STEP 13: STANDARDIZE ORDER STATUS\n")
print(f"Unique statuses BEFORE: {df['order_status'].dropna().unique().tolist()}")

df['order_status'] = df['order_status'].astype(str).str.strip().str.lower()
status_map = {
    'shipped': 'Shipped', 'shipd': 'Shipped', 'shped': 'Shipped',
    'pending': 'Pending', 'pendng': 'Pending',
    'delivered': 'Delivered', 'delvered': 'Delivered',
    'cancelled': 'Cancelled', 'cancled': 'Cancelled', 'nan': np.nan
}
df['order_status'] = df['order_status'].map(status_map)
print(f"Unique statuses AFTER: {df['order_status'].dropna().unique().tolist()}")

STEP 13: STANDARDIZE ORDER STATUS

Unique statuses BEFORE: ['Shipped', 'Cancelled', 'Delivered', 'Pending', 'SHIPPED', 'delivered', 'Shipd', 'pending', 'Delvered', 'shipped', 'cancelled', 'PENDING', 'Shped', 'DELIVERED', 'Pendng', 'Cancled', 'CANCELLED']
Unique statuses AFTER: ['Shipped', 'Cancelled', 'Delivered', 'Pending']


In [16]:
# STEP 14: STANDARDIZE DISCOUNT APPLIED
print("STEP 14: STANDARDIZE DISCOUNT APPLIED\n")
print(f"Unique values BEFORE: {df['discount_applied'].dropna().unique().tolist()}")

df['discount_applied'] = df['discount_applied'].astype(str).str.strip().str.lower()
disc_map = {
    'yes': 'Yes', 'true': 'Yes', '1': 'Yes', 'y': 'Yes',
    'no': 'No', 'false': 'No', '0': 'No', 'n': 'No', 'nan': np.nan
}
df['discount_applied'] = df['discount_applied'].map(disc_map)
print(f"Unique values AFTER: {df['discount_applied'].dropna().unique().tolist()}")
print(f"Yes: {(df['discount_applied'] == 'Yes').sum()}, No: {(df['discount_applied'] == 'No').sum()}")

STEP 14: STANDARDIZE DISCOUNT APPLIED

Unique values BEFORE: ['Yes', 'No', 'FALSE', 'N', 'no', 'NO', 'TRUE', 'yes', '1', 'Y', 'YES', '0']
Unique values AFTER: ['Yes', 'No']
Yes: 992, No: 961


In [17]:
# STEP 15: STANDARDIZE PRODUCT CATEGORY
print("STEP 15: STANDARDIZE PRODUCT CATEGORY\n")
print(f"Unique categories BEFORE: {df['product_category'].dropna().unique().tolist()}")

df['product_category'] = df['product_category'].astype(str).str.strip().str.lower()
cat_map = {
    'books': 'Books', 'boks': 'Books', 'electronics': 'Electronics', 'elec': 'Electronics',
    'electornics': 'Electronics', 'clothing': 'Clothing', 'cloth': 'Clothing', 'clothng': 'Clothing',
    'sports': 'Sports', 'spts': 'Sports', 'sport': 'Sports',
    'home & garden': 'Home & Garden', 'home and garden': 'Home & Garden', 'h&g': 'Home & Garden',
    'automotive': 'Automotive', 'toys': 'Toys', 'nan': np.nan
}
df['product_category'] = df['product_category'].map(cat_map)
print(f"Unique categories AFTER: {df['product_category'].dropna().unique().tolist()}")

STEP 15: STANDARDIZE PRODUCT CATEGORY

Unique categories BEFORE: ['Books', 'Electronics', 'Automotive', 'Sports', 'Clothing', 'Home & Garden', 'CLOTHING', 'Boks', 'Toys', 'HOME & GARDEN', 'Clothng', 'Cloth', 'SPORTS', 'Sport', 'TOYS', 'AUTOMOTIVE', '  Books  ', '  Automotive  ', 'Home and Garden', '  Electronics  ', 'BOOKS', 'H&G', '  Sports  ', '  Toys  ', 'Electornics', 'Spts', '  Home & Garden  ', 'ELECTRONICS', 'Elec', '  Clothing  ']
Unique categories AFTER: ['Books', 'Electronics', 'Automotive', 'Sports', 'Clothing', 'Home & Garden', 'Toys']


In [18]:
# STEP 16: HANDLE MISSING VALUES  (USING SCIKIT-LEARN - SHORT & CLEAN)

print("STEP 16: HANDLE MISSING VALUES (sklearn)\n")

text_cols = ['country', 'gender', 'order_status', 'discount_applied',
             'product_category', 'full_name', 'email_address', 'phone_number']
num_cols = ['purchase_amount', 'shipping_cost', 'customer_age', 'satisfaction_score_1-5']

# Capture missing counts BEFORE imputation
missing_before = df[text_cols + num_cols].isnull().sum()

# --- SCIKIT-LEARN: Categorical imputer (fill with 'Unknown') ---
cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
df[text_cols] = cat_imputer.fit_transform(df[text_cols])

# --- SCIKIT-LEARN: Numerical imputer (fill with median) ---
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Print what was filled
for col in text_cols:
    if missing_before[col] > 0:
        print(f"  {col}: {missing_before[col]} missing -> filled with 'Unknown'")
for col in num_cols:
    if missing_before[col] > 0:
        print(f"  {col}: {missing_before[col]} missing -> filled with median ({df[col].median():.2f})")

# --- Dates: random sample from existing dates (avoids fake median spike) ---
date_missing = df['registration_date'].isna().sum()
if date_missing > 0:
    df['registration_date_imputed'] = df['registration_date'].isna().astype(int)
    valid_dates = df['registration_date'].dropna().values
    random_dates = np.random.choice(valid_dates, size=date_missing, replace=True)
    df.loc[df['registration_date'].isna(), 'registration_date'] = random_dates
    print(f"  registration_date: {date_missing} missing -> filled with random sample from existing dates")
    print(f"  Flag column 'registration_date_imputed' added (1 = imputed, 0 = original)")
else:
    df['registration_date_imputed'] = 0


STEP 16: HANDLE MISSING VALUES (sklearn)

  country: 155 missing -> filled with 'Unknown'
  gender: 55 missing -> filled with 'Unknown'
  order_status: 87 missing -> filled with 'Unknown'
  discount_applied: 72 missing -> filled with 'Unknown'
  product_category: 122 missing -> filled with 'Unknown'
  full_name: 20 missing -> filled with 'Unknown'
  email_address: 774 missing -> filled with 'Unknown'
  phone_number: 229 missing -> filled with 'Unknown'
  purchase_amount: 226 missing -> filled with median (385.20)
  shipping_cost: 149 missing -> filled with median (26.55)
  customer_age: 166 missing -> filled with median (50.00)
  satisfaction_score_1-5: 228 missing -> filled with median (3.00)
  registration_date: 387 missing -> filled with random sample from existing dates
  Flag column 'registration_date_imputed' added (1 = imputed, 0 = original)


In [19]:
# STEP 17: FIX DATA TYPES
print("STEP 17: FIX DATA TYPES\n")

df['customer_age'] = df['customer_age'].round().astype(int)
df['satisfaction_score_1-5'] = df['satisfaction_score_1-5'].round().astype(int)
df = df.reset_index(drop=True)

print(f"customer_age dtype: {df['customer_age'].dtype}")
print(f"satisfaction_score_1-5 dtype: {df['satisfaction_score_1-5'].dtype}")
print(f"registration_date_imputed dtype: {df['registration_date_imputed'].dtype}")
print(f"DataFrame index reset. Rows: {len(df)}")

STEP 17: FIX DATA TYPES

customer_age dtype: int64
satisfaction_score_1-5 dtype: int64
registration_date_imputed dtype: int64
DataFrame index reset. Rows: 2025


In [20]:
# STEP 18: FINAL CHECK
print("STEP 18: FINAL CHECK\n")
print(f"Final shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Rows retained from original: {len(df)} / 2070 ({len(df)/2070*100:.1f}%)")
print("\nFirst 5 rows:")
print(df.head())

STEP 18: FINAL CHECK

Final shape: (2025, 15)
Total missing values: 0
Duplicate rows: 0
Rows retained from original: 2025 / 2070 (97.8%)

First 5 rows:
   customer_id      full_name        email_address phone_number  \
0  CUST_101181   Donald Brown              Unknown   4003496385   
1  CUST_100069   David Garcia   david.35@yahoo.com   5538535544   
2  CUST_100351  Robert Wilson  robert.50@gmail.com   5438783553   
3  CUST_101163    Donna Moore              Unknown   1793424343   
4  CUST_100429    Carol Davis   carol.43@yahoo.com   8084404666   

  registration_date  purchase_amount product_category  satisfaction_score_1-5  \
0        2024-10-20           385.20            Books                       3   
1        2020-10-27           415.43      Electronics                       1   
2        2021-10-13           385.20       Automotive                       1   
3        2019-03-23           341.33            Books                       5   
4        2023-11-06           129.95    

In [21]:
# STEP 19: SAVE CLEANED DATA
print("STEP 19: SAVE CLEANED DATA\n")
df.to_csv('cleaned_company_data.csv', index=False)
print("Cleaned data saved to 'cleaned_company_data.csv'!")
print(f"Final file contains {len(df)} rows and {len(df.columns)} columns.")

STEP 19: SAVE CLEANED DATA

Cleaned data saved to 'cleaned_company_data.csv'!
Final file contains 2025 rows and 15 columns.
